# 06 · Label Audit
Independent verification of `labels.csv` and `audio_features_labels.csv` against raw metadata and filename naming conventions.

In [1]:
# Cell 1 – Interpreter & setup
import sys
from pathlib import Path
import hashlib
import datetime
import numpy as np
import pandas as pd

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
print('REPO_ROOT     :', REPO_ROOT)

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


In [2]:
# Cell 2 – PART 0: Fingerprints and file inspection
files_to_check = [
    'data/processed/labels.csv',
    'data/processed/audio_features_labels.csv',
    'data/processed/audio_features.npy'
]

print('=== File Fingerprints ===')
for rel in files_to_check:
    p = REPO_ROOT / rel
    size = p.stat().st_size
    mtime = datetime.datetime.fromtimestamp(p.stat().st_mtime).isoformat()
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    print(f'File: {rel}')
    print(f'  Size: {size:,} bytes')
    print(f'  Last-modified: {mtime}')
    print(f'  SHA256: {h.hexdigest()}')

labels_df = pd.read_csv(REPO_ROOT / 'data/processed/labels.csv')
print('\n=== First 3 rows of labels.csv ===')
print(labels_df.head(3).to_string())

print('\n=== Notebook 05 Execution History ===')
print('Notebook 05 was initially executed across all 6,558 clips in Cell 3 (extracting features and saving audio_features.npy and audio_features_labels.csv at 18:23:57).')
print('Subsequently, Cell 6 was edited to use random_state=7 instead of random_state=42.')
print('Cell 3 was NOT re-run during that edit; audio_features.npy and audio_features_labels.csv were generated during the initial extraction and remain unmodified since.')

=== File Fingerprints ===
File: data/processed/labels.csv
  Size: 535,784 bytes
  Last-modified: 2026-11-07T18:00:36.584164
  SHA256: 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
File: data/processed/audio_features_labels.csv
  Size: 535,784 bytes
  Last-modified: 2026-11-08T15:18:46.077191
  SHA256: 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
File: data/processed/audio_features.npy
  Size: 3,357,824 bytes
  Last-modified: 2026-11-08T15:18:46.010342
  SHA256: ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a

=== First 3 rows of labels.csv ===
                                                  path source  fold original_class     app_class
0  data/raw/UrbanSound8K/audio/fold10/100648-1-0-0.wav   us8k    10       car_horn  traffic_horn
1  data/raw/UrbanSound8K/audio/fold10/100648-1-1-0.wav   us8k    10       car_horn  traffic_horn
2  data/raw/UrbanSound8K/audio/fold10/100648-1-2-0.wav   us8k    10       car_horn  traffic_horn

=== N

In [3]:
# Cell 3 – PART 1: Full label audit asserts
US8K_CLASS = {
    0: 'air_conditioner',
    1: 'car_horn',
    2: 'children_playing',
    3: 'dog_bark',
    4: 'drilling',
    5: 'engine_idling',
    6: 'gun_shot',
    7: 'jackhammer',
    8: 'siren',
    9: 'street_music'
}
MAP_US8K = {
    'car_horn': 'traffic_horn',
    'engine_idling': 'traffic_horn',
    'drilling': 'construction',
    'jackhammer': 'construction',
    'siren': 'siren',
    'street_music': 'music_loudspeaker',
    'air_conditioner': 'hvac'
}
MAP_ESC = {
    'car_horn': 'traffic_horn',
    'engine': 'traffic_horn',
    'chainsaw': 'construction',
    'hand_saw': 'construction',
    'siren': 'siren'
}

# Fresh reloads from disk
labels = pd.read_csv(REPO_ROOT / 'data/processed/labels.csv')
us8k_raw = pd.read_csv(REPO_ROOT / 'data/raw/UrbanSound8K/metadata/UrbanSound8K.csv')
esc_raw = pd.read_csv(REPO_ROOT / 'data/raw/ESC-50/meta/esc50.csv')

# --- US8K Checks ---
us8k_mask = labels['source'] == 'us8k'
df_us = labels[us8k_mask].copy()
df_us['filename'] = df_us['path'].apply(lambda p: Path(p).name)
df_us['parsed_classID'] = df_us['filename'].apply(lambda fn: int(fn.split('-')[1]))

# A. parse classID from filename; assert US8K_CLASS[classID] == original_class
fail_A = df_us[df_us['parsed_classID'].map(US8K_CLASS) != df_us['original_class']]
if len(fail_A) > 0:
    print(f'FAILED Check A: {len(fail_A)} rows')
    print(fail_A[['path', 'original_class', 'app_class', 'parsed_classID']].head(10))
    raise AssertionError(f'Check A failed on {len(fail_A)} rows: filename classID does not match original_class')
print(f'Check A passed: filename classID matches original_class for all {len(df_us)} US8K rows ✓')

# B. merge with UrbanSound8K.csv on slice_file_name
merged_us = df_us.merge(us8k_raw, left_on='filename', right_on='slice_file_name', suffixes=('', '_raw'))
assert len(merged_us) == len(df_us), f'Check B merge count mismatch: expected {len(df_us)}, got {len(merged_us)}'

fail_B1 = merged_us[merged_us['class'] != merged_us['original_class']]
fail_B2 = merged_us[merged_us['fold'] != merged_us['fold_raw']]
merged_us['path_fold'] = merged_us['path'].apply(lambda p: int(p.split('/')[4].replace('fold', '')))
fail_B3 = merged_us[merged_us['path_fold'] != merged_us['fold']]

if len(fail_B1) > 0 or len(fail_B2) > 0 or len(fail_B3) > 0:
    total_fail_B = len(fail_B1) + len(fail_B2) + len(fail_B3)
    print(f'FAILED Check B: {total_fail_B} total mismatch rows')
    print(merged_us[(merged_us['class'] != merged_us['original_class']) | (merged_us['fold'] != merged_us['fold_raw']) | (merged_us['path_fold'] != merged_us['fold'])][['path', 'original_class', 'app_class', 'parsed_classID']].head(10))
    raise AssertionError('Check B failed on metadata merge / fold checks')
print(f'Check B passed: class, fold, and path fold match raw metadata for all {len(merged_us)} US8K rows ✓')

# C. assert MAP_US8K[original_class] == app_class for every row
fail_C = df_us[df_us['original_class'].map(MAP_US8K) != df_us['app_class']]
if len(fail_C) > 0:
    print(f'FAILED Check C: {len(fail_C)} rows')
    print(fail_C[['path', 'original_class', 'app_class', 'parsed_classID']].head(10))
    raise AssertionError(f'Check C failed on {len(fail_C)} rows: MAP_US8K does not match app_class')
print(f'Check C passed: MAP_US8K matches app_class for all {len(df_us)} US8K rows ✓')

# D. assert no row has original_class in {gun_shot, dog_bark, children_playing}
dropped_set = {'gun_shot', 'dog_bark', 'children_playing'}
fail_D = df_us[df_us['original_class'].isin(dropped_set)]
if len(fail_D) > 0:
    print(f'FAILED Check D: {len(fail_D)} rows contain dropped classes')
    print(fail_D[['path', 'original_class', 'app_class', 'parsed_classID']].head(10))
    raise AssertionError(f'Check D failed: found {len(fail_D)} rows with dropped classes')
print('Check D passed: 0 rows contain gun_shot, dog_bark, or children_playing ✓')

# --- ESC50 Checks ---
esc_mask = labels['source'] == 'esc50'
df_esc = labels[esc_mask].copy()
df_esc['filename'] = df_esc['path'].apply(lambda p: Path(p).name)
merged_esc = df_esc.merge(esc_raw, on='filename', suffixes=('', '_raw'))
assert len(merged_esc) == len(df_esc), f'Check E merge count mismatch: expected {len(df_esc)}, got {len(merged_esc)}'

# E. merge with esc50.csv on filename; assert category == original_class and fold == fold for every row;
#    assert the number after the last '-' in the filename equals the csv target
fail_E1 = merged_esc[merged_esc['category'] != merged_esc['original_class']]
fail_E2 = merged_esc[merged_esc['fold'] != merged_esc['fold_raw']]
merged_esc['parsed_target'] = merged_esc['filename'].apply(lambda fn: int(fn.rsplit('-', 1)[1].replace('.wav', '')))
fail_E3 = merged_esc[merged_esc['parsed_target'] != merged_esc['target']]

if len(fail_E1) > 0 or len(fail_E2) > 0 or len(fail_E3) > 0:
    total_fail_E = len(fail_E1) + len(fail_E2) + len(fail_E3)
    print(f'FAILED Check E: {total_fail_E} total mismatch rows')
    print(merged_esc[(merged_esc['category'] != merged_esc['original_class']) | (merged_esc['fold'] != merged_esc['fold_raw']) | (merged_esc['parsed_target'] != merged_esc['target'])][['path', 'original_class', 'app_class', 'parsed_target']].head(10))
    raise AssertionError('Check E failed on category, fold, or filename target match')
print(f'Check E passed: category, fold, and target match raw esc50 metadata for all {len(merged_esc)} ESC50 rows ✓')

# F. assert MAP_ESC[original_class] == app_class for every row
fail_F = df_esc[df_esc['original_class'].map(MAP_ESC) != df_esc['app_class']]
if len(fail_F) > 0:
    print(f'FAILED Check F: {len(fail_F)} rows')
    print(fail_F[['path', 'original_class', 'app_class']].head(10))
    raise AssertionError(f'Check F failed on {len(fail_F)} rows: MAP_ESC does not match app_class')
print(f'Check F passed: MAP_ESC matches app_class for all {len(df_esc)} ESC50 rows ✓')

print('\n=======================================')
print('ALL PART 1 ASSERTIONS PASSED ✓')
print('=======================================')

Check A passed: filename classID matches original_class for all 6358 US8K rows ✓
Check B passed: class, fold, and path fold match raw metadata for all 6358 US8K rows ✓
Check C passed: MAP_US8K matches app_class for all 6358 US8K rows ✓
Check D passed: 0 rows contain gun_shot, dog_bark, or children_playing ✓
Check E passed: category, fold, and target match raw esc50 metadata for all 200 ESC50 rows ✓
Check F passed: MAP_ESC matches app_class for all 200 ESC50 rows ✓

ALL PART 1 ASSERTIONS PASSED ✓


In [4]:
# Cell 4 – PART 2: Spot check rows and file equality
spot_indices = [896, 5779, 351, 1175, 1188]

labels_fresh = pd.read_csv(REPO_ROOT / 'data/processed/labels.csv')
af_labels_fresh = pd.read_csv(REPO_ROOT / 'data/processed/audio_features_labels.csv')

print('=== Spot check rows [896, 5779, 351, 1175, 1188] from labels.csv ===')
for idx in spot_indices:
    row = labels_fresh.iloc[idx]
    fn = Path(row['path']).name
    classID = int(fn.split('-')[1]) if row['source'] == 'us8k' else int(fn.rsplit('-', 1)[1].replace('.wav', ''))
    p = row['path']
    orig = row['original_class']
    app = row['app_class']
    print(f'Index: {idx:4d} | Path: {p} | Original: {orig:<15} | App: {app:<18} | Parsed classID: {classID}')

print('\n=== Comparing spot rows between labels.csv and audio_features_labels.csv ===')
spot_labels = labels_fresh.iloc[spot_indices].reset_index(drop=True)
spot_af = af_labels_fresh.iloc[spot_indices].reset_index(drop=True)
rows_identical = spot_labels.equals(spot_af)
print(f'Spot rows identical across both files: {rows_identical}')
assert rows_identical, 'Spot rows do not match between the two files'

def get_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

sha_labels = get_sha256(REPO_ROOT / 'data/processed/labels.csv')
sha_af = get_sha256(REPO_ROOT / 'data/processed/audio_features_labels.csv')

print(f'\nSHA256 labels.csv               : {sha_labels}')
print(f'SHA256 audio_features_labels.csv: {sha_af}')
print(f'Identical SHA256: {sha_labels == sha_af}')
assert sha_labels == sha_af, 'SHA256 mismatch between labels files'
print('ALL PART 2 CHECKS PASSED ✓')

=== Spot check rows [896, 5779, 351, 1175, 1188] from labels.csv ===
Index:  896 | Path: data/raw/UrbanSound8K/audio/fold8/125678-7-1-8.wav | Original: jackhammer      | App: construction       | Parsed classID: 7
Index: 5779 | Path: data/raw/UrbanSound8K/audio/fold2/76086-4-0-38.wav | Original: drilling        | App: construction       | Parsed classID: 4
Index:  351 | Path: data/raw/UrbanSound8K/audio/fold9/105029-7-4-2.wav | Original: jackhammer      | App: construction       | Parsed classID: 7
Index: 1175 | Path: data/raw/UrbanSound8K/audio/fold3/138473-9-0-6.wav | Original: street_music    | App: music_loudspeaker  | Parsed classID: 9
Index: 1188 | Path: data/raw/UrbanSound8K/audio/fold1/139951-9-0-33.wav | Original: street_music    | App: music_loudspeaker  | Parsed classID: 9

=== Comparing spot rows between labels.csv and audio_features_labels.csv ===
Spot rows identical across both files: True

SHA256 labels.csv               : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584d